In [1]:
import re
import pandas as pd

# Load the dataset
df = pd.read_csv("train.csv")

print("Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFirst Tweet Sample:")
print(df['tweet'].iloc[0])

Dataset Shape: (24783, 6)
Columns: ['count', 'hate_speech_count', 'offensive_language_count', 'neither_count', 'class', 'tweet']

First Tweet Sample:
!!! RT @mayasolovely: As a woman you shouldn't complain about cleaning up your house. &amp; as a man you should always take the trash out...


In [5]:
# 1. Define Common Affixes and Rules
PREFIXES = ['un', 're', 'dis', 'pre', 'over', 'under', 'mis', 'im', 'in', 'non', 'anti', 'sub']

SUFFIXES = [
    ('ing', 'Present Participle / Gerund'),
    ('ed', 'Past Tense / Past Participle'),
    ('es', 'Plural Noun / 3rd Person Present Verb'),
    ('s', 'Plural Noun / 3rd Person Present Verb'),
    ('ly', 'Adverbial Suffix'),
    ('er', 'Agent Noun / Comparative Adjective'),
    ('est', 'Superlative Adjective'),
    ('ment', 'Nominal Suffix (Noun)'),
    ('ness', 'Abstract Noun Suffix'),
    ('able', 'Adjectival Suffix'),
    ('ful', 'Adjectival Suffix'),
    ('less', 'Adjectival Suffix')
]

# Irregular Morphological Mappings
IRREGULAR_NOUNS = {'woman': 'women', 'man': 'men', 'child': 'children', 'person': 'people'}
IRREGULAR_VERBS = {
    'take': {'3s': 'takes', 'past': 'took', 'past_part': 'taken', 'ing': 'taking'},
    'cry': {'3s': 'cries', 'past': 'cried', 'past_part': 'cried', 'ing': 'crying'},
    'be': {'3s': 'is', 'past': 'was/were', 'past_part': 'been', 'ing': 'being'}
}

# 2. Token Cleaning
def clean_and_tokenize(text):
    text = re.sub(r'http\S+|www\S+|\S+@\S+', '', text) # Remove URLs and handles
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)           # Keep only alphabets
    return text.split()

# 3. Morphological Analyzer Function
def analyze_morphology(word):
    w_lower = word.lower()
    prefix, suffix, suffix_desc = "None", "None", "Base Form / Uninflected"
    root = w_lower

    # Check Prefix
    for p in PREFIXES:
        if w_lower.startswith(p) and len(w_lower) > len(p) + 2:
            prefix = p
            root = root[len(p):]
            break

    # Check Suffix
    for s, desc in SUFFIXES:
        if root.endswith(s) and len(root) > len(s) + 2:
            suffix = s
            suffix_desc = desc
            root = root[:-len(s)]
            break

    return {
        'Word': word,
        'Prefix': prefix,
        'Root Base': root,
        'Suffix': suffix,
        'Morphological Function': suffix_desc
    }

# 4. Word Generation Engine Function
def generate_words(root):
    root = root.lower()
    
    # Verb Inflections
    if root in IRREGULAR_VERBS:
        v = IRREGULAR_VERBS[root]
        verb_forms = f"3rd Pres: {v['3s']} | Past: {v['past']} | Pres.Part: {v['ing']}"
    else:
        if root.endswith('e'):
            verb_forms = f"3rd Pres: {root}s | Past: {root}d | Pres.Part: {root[:-1]}ing"
        elif root.endswith('y') and len(root) > 1 and root[-2] not in 'aeiou':
            verb_forms = f"3rd Pres: {root[:-1]}ies | Past: {root[:-1]}ied | Pres.Part: {root}ing"
        else:
            verb_forms = f"3rd Pres: {root}s | Past: {root}ed | Pres.Part: {root}ing"

    # Plural Nouns
    plural = IRREGULAR_NOUNS.get(root, root + 'es' if root.endswith(('s', 'sh', 'ch', 'x')) else root + 's')
    
    # Derivations
    agent = root + 'r' if root.endswith('e') else root + 'er'
    adverb = root[:-1] + 'ily' if root.endswith('y') else root + 'ly'

    return {
        'Root': root,
        'Plural Noun': plural,
        'Verb Inflections': verb_forms,
        'Agent Noun': agent,
        'Adverb Form': adverb
    }

In [4]:
# Select a sample tweet from dataset
sample_tweet = df['tweet'].iloc[0]
tokens = clean_and_tokenize(sample_tweet)

# Perform analysis
morph_results = [analyze_morphology(token) for token in tokens]
morph_df = pd.DataFrame(morph_results)

print("--- RAW TEXT ---")
print(sample_tweet)

print("\n--- MORPHOLOGICAL ANALYSIS RESULT ---")
morph_df

--- RAW TEXT ---
!!! RT @mayasolovely: As a woman you shouldn't complain about cleaning up your house. &amp; as a man you should always take the trash out...

--- MORPHOLOGICAL ANALYSIS RESULT ---


,Word,Prefix,Root Base,Suffix,Morphological Function
0,RT,None,rt,None,Base Form / Uninflected
1,mayasolovely,None,mayasolove,ly,Adverbial Suffix
2,As,None,as,None,Base Form / Uninflected
3,a,None,a,None,Base Form / Uninflected
4,woman,None,woman,None,Base Form / Uninflected
5,you,None,you,None,Base Form / Uninflected
6,shouldn,None,shouldn,None,Base Form / Uninflected
7,t,None,t,None,Base Form / Uninflected
8,complain,None,complain,None,Base Form / Uninflected
9,about,None,about,None,Base Form / Uninflected


In [6]:
# Select key base roots extracted from the dataset
target_roots = ['clean', 'complain', 'woman', 'house', 'cry', 'take', 'trash']

# Generate new word forms
gen_results = [generate_words(r) for r in target_roots]
gen_df = pd.DataFrame(gen_results)

print("--- WORD GENERATION ENGINE OUTPUT ---")
gen_df

--- WORD GENERATION ENGINE OUTPUT ---


,Root,Plural Noun,Verb Inflections,Agent Noun,Adverb Form
0,clean,cleans,3rd Pres: cleans | Past: cleaned | Pres.Part: ...,cleaner,cleanly
1,complain,complains,3rd Pres: complains | Past: complained | Pres....,complainer,complainly
2,woman,women,3rd Pres: womans | Past: womaned | Pres.Part: ...,womaner,womanly
3,house,houses,3rd Pres: houses | Past: housed | Pres.Part: h...,houser,housely
4,cry,crys,3rd Pres: cries | Past: cried | Pres.Part: crying,cryer,crily
5,take,takes,3rd Pres: takes | Past: took | Pres.Part: taking,taker,takely
6,trash,trashes,3rd Pres: trashs | Past: trashed | Pres.Part: ...,trasher,trashly
